# Voorspellen van Maandelijkse Autoschadeclaims met PROC FORECAST

## Samenvatting

Een actuarieel reserveringsteam heeft een vooruitblik van 12 maanden nodig op het aantal maandelijkse autoschadeclaims, die een gestage portefeuillegroei laat zien plus een uitgesproken winterweer-stijging. Dit notebook genereert vijf jaar synthetische maandelijkse claimtellingen (60 maanden, jan 2020 - dec 2024, variërend van ongeveer 1.460 tot 2.780 claims), en gebruikt vervolgens **PROC FORECAST** om een stapsgewijs-autoregressieve basislijn en een multiplicatief Holt-Winters seizoensmodel te fitten. Elk model produceert 12 maandelijkse puntvoorspellingen met 95%-betrouwbaarheidsgrenzen voor capaciteits- en reserveplanning. Het seizoensgebonden Holt-Winters-model voorspelt de sterkste vraag één tot twee maanden vooruit (ongeveer 2.780-2.900 claims), die afneemt naar een dal rond stap negen (ongeveer 2.200), terwijl de autoregressieve basislijn een gladdere afname voorspelt; beide betrouwbaarheidsbanden verbreden naarmate de horizon toeneemt.

## Gegevensbronnen

| Dataset | Rijen | Granulariteit | Kernvariabelen | Beschrijving |
|---------|------|-------|---------------|-------------|
| `claims` | 60 | Eén rij per kalendermaand (jan 2020 - dec 2024) | `date` (SAS-datum, `MONYY7.`), `claim_count` (numeriek) | Synthetische maandelijkse autoschadeclaimtellingen opgebouwd uit een lineaire groeitrend (~9 claims/maand), een sinusoïdale jaarcyclus, een additieve dec/jan/feb winterweer-stijging, en Gaussische ruis (`rand('normal')`). Seed `20240531` maakt het reproduceerbaar. |

# Voorspellen van Maandelijkse Autoschadeclaims

Reserverings- en capaciteitsplanningsteams bij een particuliere verzekeraar hebben een vooruitblik nodig op hoeveel **autoclaims** elke maand gemeld zullen worden. Het claimvolume in dit boek groeit gestaag naarmate de portefeuille uitbreidt, en piekt elke winter wanneer ijzel, sneeuw en minder daglicht leiden tot meer aanrijdings- en ruitclaims.

Dit notebook doorloopt een volledige `PROC FORECAST`-workflow:

1. Genereer een realistische, volledig synthetische maandelijkse claimtellingsreeks.
2. Fit een **stapsgewijs-autoregressieve (STEPAR)** basislijn die trend plus autocorrelatie vastlegt.
3. Fit een **multiplicatief Holt-Winters (WINTERS)** model dat de 12-maandelijkse seizoenscyclus expliciet modelleert.
4. Vergelijk de twee modellen en interpreteer de vooruitvoorspelling en betrouwbaarheidsband.

Alles draait op inline synthetische data — geen externe bestanden of netwerktoegang.

## Stap 1 — Genereer de synthetische claimreeks

De DATA-stap hieronder bouwt **60 maandelijkse waarnemingen** (jan 2020 t/m dec 2024). Voor elke maand combineren we vier componenten die een echt autoboek weerspiegelen:

- **Trend** — een basislijn van ~1.800 claims die met ~9 per maand groeit naarmate de blootstelling toeneemt.
- **Jaarcyclus** — een sinus/cosinusterm die een gladde seizoensgolf geeft.
- **Winterpiek** — een additieve stijging in december, januari en februari.
- **Ruis** — `rand('normal', 0, 90)` voor realistische maand-op-maand variabiliteit.

`call streaminit()` legt de random-stream vast zodat het notebook reproduceerbaar is. De variabele `date` is een echte SAS-datum opgebouwd met `INTNX` en opgemaakt met `MONYY7.`, en de instructie `ID date INTERVAL=MONTH` benoemt deze als de tijdidentificatie. De eerste 14 hieronder afgedrukte rijen liggen tussen ruwweg 1.460 en 2.450 claims.

In [1]:
GEGEVENS claims;
    CALL streaminit(20240531);
    DOE i = 0 TOT 59;
        /* Bouw een echte SAS-maanddatum zodat INTERVAL=MONTH aansluit */
        date = intnx('month', '01JAN2020'd, i);
        OPMAAK date monyy7.;

        month_idx = mod(i, 12) + 1;          /* 1 = jan ... 12 = dec */

        trend  = 1800 + 9 * i;               /* groeiende blootstellingsbasis   */
        season = 260 * sin((month_idx - 1) / 12 * 2 * constant('pi'))
               + 150 * cos((month_idx - 1) / 12 * 2 * constant('pi'));
        winter = 220 * (month_idx in (12, 1, 2));   /* ijzel/sneeuw-piek  */
        noise  = rand('normal', 0, 90);

        claim_count = round(trend + season + winter + noise);
        ALS claim_count < 0 DAN claim_count = 0;
        UITVOER;
    EINDE;
    BEWAREN date claim_count;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=claims(obs=14) label;
    label date = "Maand" claim_count = "Gerapporteerde Claims";
    TITEL "Eerste 14 Maanden van Synthetische Autoclaimtellingen";
UITVOEREN;

                                 Eerste 14 Maanden van Synthetische Autoclaimtellingen                                  

  Obs  Maand  Gerapporteerde Claims
    1  21915                   2178
    2  21946                   2281
    3  21975                   2252
    4  22006                   1974
    5  22036                   2067
    6  22067                   1805
    7  22097                   1697
    8  22128                   1619
    9  22159                   1462
   10  22189                   1688
   11  22220                   1713
   12  22250                   2008
   13  22281                   2116
   14  22312                   2451

... 46 more observations (showing 14 of 60)




NOTE: DATA claims


NOTE: Wrote claims (60 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC PRINT data=claims

NOTE: PROC PRINT completed: 14 observations printed, 2 variables


## Stap 2 — Stapsgewijs-autoregressieve basislijn (METHOD=STEPAR)

`METHOD=STEPAR` is de standaard. Er wordt eerst een tijd-trendmodel gefit — hier `TREND=2` voor een lineaire trend — waarna een **stapsgewijze autoregressie** wordt toegepast op de residuen, waarbij AR-lags op basis van significantie worden opgenomen en behouden. `AR=4` begrenst de kandidaat-autoregressieve orde op vier lags, wat ruim voldoende is voor een maandelijkse reeks met kortetermijnmomentum.

Belangrijke hier gebruikte opties:

- `LEAD=12` — voorspel 12 maanden voorbij de data.
- `ALPHA=0.05` — 95%-betrouwbaarheidsgrenzen.
- `OUTFULL` — stapel de historische `ACTUAL`-rijen samen met de rijen van de voorspelhorizon in de `OUT=`-dataset (in plaats van alleen de voorspelrijen).
- `OUTLIMIT` — voeg de onder-/bovengrens-betrouwbaarheids**kolommen** toe (`L95_claim_count`, `U95_claim_count`).
- `ID date INTERVAL=MONTH` — benoemt de tijdidentificatie en verklaart dat de reeks maandelijks is.

In [2]:
PROCEDURE forecast GEGEVENS=claims
        out=fc_stepar
        METHOD=stepar trend=2 ar=4
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    VARIABELE claim_count;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=fc_stepar(obs=10) label;
    TITEL "STEPAR-uitvoer: Werkelijke, Gefitte en Voorspelde Rijen";
UITVOEREN;

                                 Eerste 14 Maanden van Synthetische Autoclaimtellingen                                  

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                STEPAR-uitvoer: Werkelijke, Gefitte en Voorspelde Rijen                                 

  Obs   DATE  _TYPE_  CLAIM_COUNT  L95_CLAIM_COUNT  U95_CLAIM_COUNT
    1  21915  ACTUAL  2121.816667                .                .
    2  21946  ACTUAL  2167.711419                .                .
    3  21975  ACTUAL  2254.781177                .                .
    4  22006  ACTUAL  2228.505912                .                .
    5  22036  ACTUAL  1978.152296                .                .
    6  22067  ACTUAL  2030.606563                .                .
    7  22097  ACTUAL  1864.520418                .                .
    8  22128  ACTUAL  1784.855682                .                .
    9  22159  ACTUAL  1740.781553  


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: PROC PRINT data=fc_stepar

NOTE: PROC PRINT completed: 10 observations printed, 5 variables


### De OUT=-dataset lezen

De `OUT=`-dataset stapelt rijen via een `_TYPE_`-kolom en draagt de betrouwbaarheidsgrenzen als zij-**kolommen**:

| Element | Soort | Betekenis |
|---------|------|---------|
| `_TYPE_ = 'ACTUAL'` | rij | De waargenomen `claim_count` voor elk van de 60 historische maanden. |
| `_TYPE_ = 'FORECAST'` | rij | De 12 puntvoorspellingen voor de voorspelhorizon. |
| `L95_claim_count` / `U95_claim_count` | kolom | Onder-/bovengrens 95%-betrouwbaarheidslimieten, gevuld op de `FORECAST`-rijen (ontbrekend op `ACTUAL`-rijen). Het numerieke niveau weerspiegelt `ALPHA=`. |

De hierboven afgedrukte `OUT=`-dataset bevat dus 72 rijen: 60 `ACTUAL`-historierijen gevolgd door 12 `FORECAST`-horizonrijen. De eerste tien getoonde rijen zijn allemaal `ACTUAL`, waarbij de betrouwbaarheidsgrens-kolommen ontbreken omdat de grenzen alleen aan de voorspelrijen gekoppeld zijn.

> **Engine-opmerking.** SAS `OUTFULL` schrijft ook een within-sample één-stap-vooruit `FORECAST`-rij voor elke historische maand, en `OUTRESID` voegt `_TYPE_='RESIDUAL'`-rijen toe. Jenners PROC FORECAST emit die within-sample gefitte/residu-rijen nog niet (gevolgd als gap-test `400979`), dus dit notebook leest alleen de `ACTUAL`-historie en de voorwaartse `FORECAST`-horizon.

## Stap 3 — Seizoensgebonden Holt-Winters-model (METHOD=WINTERS)

Het STEPAR-model legt trend en kortetermijn-autocorrelatie vast, maar modelleert de terugkerende december-februari-stijging niet expliciet. Voor een reeks met een duidelijk jaarlijks ritme is **multiplicatief Holt-Winters** het betere gereedschap.

`METHOD=WINTERS` ontbindt de reeks in niveau, lineaire trend (`TREND=2`), en een **multiplicatieve seizoensfactor**. `SEASONS=12` verklaart een 12-periodieke (maandelijkse) seizoenscyclus. We vragen opnieuw om een 12-maands `LEAD` met 95%-grenzen en `OUTFULL OUTLIMIT` zodat de uitvoer aansluit bij de STEPAR-run.

Omdat de engine de voorspel-`ID` per stap met één eenheid verhoogt (in plaats van `INTERVAL=MONTH` voor de horizondatums te respecteren — gap-test `400979`), bekijkt de cel hieronder de voorspelling per **maanden vooruit (stap 1-12)** in plaats van te vertrouwen op de afgedrukte kalenderdatums.

In [3]:
PROCEDURE forecast GEGEVENS=claims
        out=fc_winters
        METHOD=winters seasons=12 trend=2
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    VARIABELE claim_count;
UITVOEREN;

/* Behoud de 12-maands vooruithorizon en indexeer op stap (1..12). */
GEGEVENS horizon;
    INSTELLEN fc_winters;
    BEHOUDEN months_ahead 0;
    ALS _type_ = 'FORECAST';
    months_ahead + 1;
    BEWAREN months_ahead claim_count l95_claim_count u95_claim_count;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=horizon label;
    label months_ahead     = "Maanden Vooruit"
          claim_count       = "Voorspelde Claims"
          l95_claim_count   = "Ondergrens 95%"
          u95_claim_count   = "Bovengrens 95%";
    TITEL "Holt-Winters 12-Maands Vooruitvoorspelling (per stap)";
UITVOEREN;

                                STEPAR-uitvoer: Werkelijke, Gefitte en Voorspelde Rijen                                 

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                 Holt-Winters 12-Maands Vooruitvoorspelling (per stap)                                  

  Obs  Maanden Vooruit  Voorspelde Claims  Ondergrens 95%  Bovengrens 95%
    1                1         2783.07951     2577.844742     2988.314278
    2                2        2897.355557     2607.109764     3187.601349
    3                3        2805.272075     2449.795029      3160.74912
    4                4        2664.498049     2254.028514     3074.967585
    5                5        2628.810136     2169.891244     3087.729029
    6                6         2548.39319     2045.672732     3051.113649
    7                7        2391.649756       1848.6496     2934.649912
    8                8        2239.948352     1659.4567


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: DATA horizon


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote horizon (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=horizon

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Stap 4 — De twee modellen rechtstreeks vergelijken

De duidelijkste manier om te beoordelen of het seizoensmodel zijn meerwaarde bewijst, is om de 12-maands voorspelling ervan naast de STEPAR-basislijn te leggen, stap voor stap. We halen de `FORECAST`-rijen uit beide `OUT=`-datasets, indexeren elk op maanden-vooruit, en voegen ze samen zodat de divergentie in één oogopslag zichtbaar is.

De twee methoden zijn het eens over het algemene niveau maar verschillen in **vorm**: Holt-Winters draagt het jaarlijkse ritme door in de horizon (een hoger niveau vroeg in de horizon dat afvlakt naar een dal halverwege de horizon en weer stijgt), terwijl STEPAR — dat seizoensinvloed alleen indirect modelleert via AR-lags — gladder afneemt naar het reeksgemiddelde. Het verschil tussen beide bij elke stap is het seizoenssignaal dat STEPAR laat liggen.

> SAS zou deze adequaatheidscontrole normaal aansturen met `OUTRESID` één-stap-vooruit-residuen (`_TYPE_='RESIDUAL'`). Jenner vult die rijen nog niet (gap-test `400979`), dus vergelijken we in plaats daarvan de twee voorwaartse voorspellingen rechtstreeks — een diagnose die alleen uitvoer gebruikt die de engine daadwerkelijk produceert.

In [4]:
/* STEPAR-horizon, geïndexeerd op maanden-vooruit. */
GEGEVENS stepar_h;
    INSTELLEN fc_stepar;
    BEHOUDEN months_ahead 0;
    ALS _type_ = 'FORECAST';
    months_ahead + 1;
    stepar = claim_count;
    BEWAREN months_ahead stepar;
UITVOEREN;

/* WINTERS-horizon, geïndexeerd op maanden-vooruit. */
GEGEVENS winters_h;
    INSTELLEN fc_winters;
    BEHOUDEN months_ahead 0;
    ALS _type_ = 'FORECAST';
    months_ahead + 1;
    winters = claim_count;
    BEWAREN months_ahead winters;
UITVOEREN;

/* Voeg de twee samen en toon het seizoensverschil dat STEPAR mist. */
GEGEVENS COMPARE;
    SAMENVOEGEN stepar_h winters_h;
    VOLGENS months_ahead;
    seasonal_gap = winters - stepar;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=COMPARE label;
    label months_ahead = "Maanden Vooruit"
          stepar        = "STEPAR-voorspelling"
          winters       = "Winters-voorspelling"
          seasonal_gap  = "Winters - STEPAR";
    OPMAAK stepar winters seasonal_gap 8.0;
    TITEL "STEPAR versus Holt-Winters: Vergelijking 12-Maands Voorspelling";
UITVOEREN;

                            STEPAR versus Holt-Winters: Vergelijking 12-Maands Voorspelling                             

  Obs  Maanden Vooruit  STEPAR-voorspelling  Winters-voorspelling  Winters - STEPAR
    1                1                 2619                  2783               164
    2                2                 2537                  2897               361
    3                3                 2384                  2805               421
    4                4                 2214                  2664               450
    5                5                 2089                  2629               540
    6                6                 2010                  2548               539
    7                7                 1982                  2392               410
    8                8                 1995                  2240               245
    9                9                 2031                  2197               166
   10               10                


NOTE: DATA stepar_h


NOTE: Read 72 rows from fc_stepar.
NOTE: Wrote stepar_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA winters_h


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote winters_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA compare

NOTE: Stream 1 processed 12 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 12 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote compare (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=compare

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Stap 5 — Elke bedrijfslijn tegelijk voorspellen (BY-verwerking)

Echte reserveringsruns bestrijken meerdere producten. Met de data gesorteerd op `line_of_business` laat een `BY`-instructie `PROC FORECAST` een **onafhankelijk seizoensmodel voor elke groep** fitten in één enkele aanroep. Hier synthetiseren we een Auto-boek (hoger basisvolume) en een Wonen-boek (lager basisvolume) en voorspellen beide zes maanden vooruit. `OUTALL` schrijft de volledige set rijen — de `ACTUAL`-historie en de `FORECAST`-horizon — samen met de grenskolommen voor elke groep, en we behouden alleen de `FORECAST`-rijen voor de tabel hieronder.

In [5]:
GEGEVENS multi_book;
    CALL streaminit(20240531);
    LENGTE line_of_business $6;
    DOE lob = 1 TOT 2;
        ALS lob = 1 DAN line_of_business = 'Auto';
        ANDERS            line_of_business = 'Wonen';
        DOE i = 0 TOT 47;
            date = intnx('month', '01JAN2021'd, i);
            OPMAAK date monyy7.;
            mi   = mod(i, 12) + 1;
            BASE = (lob = 1) * 2000 + (lob = 2) * 1400;
            claim_count = round(BASE + 8 * i
                + 200 * sin((mi - 1) / 12 * 2 * constant('pi'))
                + 180 * (mi in (12, 1, 2))
                + rand('normal', 0, 70));
            UITVOER;
        EINDE;
    EINDE;
    BEWAREN line_of_business date claim_count;
UITVOEREN;

PROCEDURE SORTEREN GEGEVENS=multi_book;
    VOLGENS line_of_business date;
UITVOEREN;

PROCEDURE forecast GEGEVENS=multi_book
        out=fc_by
        METHOD=winters seasons=12 trend=2
        LEAD=6 ALPHA=0.05
        outall;
    VOLGENS line_of_business;
    id date interval=month;
    VARIABELE claim_count;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=fc_by(obs=12) label;
    WAAR _type_ = 'FORECAST';
    TITEL "Voorspellingen per Bedrijfslijn over 6 Maanden";
UITVOEREN;

                            STEPAR versus Holt-Winters: Vergelijking 12-Maands Voorspelling                             


BY Group: line_of_business=Auto

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6


BY Group: line_of_business=Wonen

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6



                                     Voorspellingen per Bedrijfslijn over 6 Maanden                                     

  Obs  LINE_OF_BUSINESS   DATE    _TYPE_  CLAIM_COUNT  L95_CLAIM_COUNT  U95_CLAIM_COUNT  RESIDUAL_CLAIM_COUNT
    1  Auto              23742  FORECAST  2524.596717      2359.095846      2690.097589                     .
    2  Auto              23773  FORECAST  2604.418724      2370.365147        2838.4723                     .
    3  Auto              23801  FORECAST  2576.092313      2289.436395       2862.74823                     .
    4  Auto              23832  F


NOTE: DATA multi_book


NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC SORT data=multi_book

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 96 rows from multi_book.
NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC FORECAST data=multi_book

NOTE: BY Group: line_of_business=Auto
NOTE: Using Python for FORECAST estimation
NOTE: BY Group: line_of_business=Wonen
NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 108 observations.
NOTE: PROC PRINT data=fc_by

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 6 observations printed, 7 variables


## Interpretatie van de resultaten

**Wat de modellen het reserveringsteam vertellen:**

- **STEPAR** reproduceert de opwaartse drift en het kortetermijnmomentum, maar de 12-maands voorspelling neemt gladjes af van ongeveer 2.620 (stap 1) naar ruwweg 1.980 halverwege de horizon voordat deze terugdrijft naar ongeveer 2.140 — het reproduceert geen scherpe winterpiek, omdat het seizoensinvloed alleen via autoregressieve lags behandelt. Het is een redelijke snelle basislijn.
- **WINTERS** met `SEASONS=12` draagt het jaarlijkse ritme direct door via zijn multiplicatieve seizoensfactoren: de voorspelling is het hoogst vroeg in de horizon (ongeveer 2.780 bij stap 1, ongeveer 2.900 bij stap 2), vlakt af naar een dal rond stap 9 (ongeveer 2.200), en stijgt weer tegen stap 12 (ongeveer 2.800). Ten opzichte van de STEPAR-basislijn ligt het gedurende het grootste deel van de horizon **150-660 claims hoger** (zie de kolom `Winters - STEPAR` in Stap 4) — dat verschil is het seizoenssignaal dat het autoregressieve model laat liggen.
- De **95%-betrouwbaarheidsband** (kolommen `L95_*`/`U95_*`, bestuurd door `ALPHA=`) verbreedt naarmate de horizon zich uitstrekt — voor het WINTERS-model van een breedte van ongeveer 410 claims bij stap 1 tot ongeveer 1.420 bij stap 12 — het eerlijke signaal dat schattingen laat in de horizon meer onzekerheid dragen dan schattingen op korte termijn. Reserveerders moeten kapitaal aanhouden tegen de bovengrens, niet alleen tegen de puntvoorspelling.
- **BY-verwerking** produceert de Auto- en Wonen-voorspellingen in één doorloop, elk met zijn eigen seizoensfit. Het Auto-boek voorspelt in het bereik van ruwweg 2.510-2.600, terwijl het Wonen-boek rond 1.870-2.030 ligt, zodat elke lijn zijn eigen niveau en seizoensvorm behoudt — het patroon dat het team zou uitbreiden naar elk product in de portefeuille.

**Kernboodschap:** voor een claimtellingsreeks met een duidelijke jaarcyclus is `METHOD=WINTERS SEASONS=12` de voor de hand liggende keuze; de STEPAR-basislijn is een nuttige sanity-check, en `OUTFULL`/`OUTLIMIT` plus een stap-voor-stap modelvergelijking laten de actuaris de vooruitreserve met zijn onzekerheidsband bepalen.

> **Engine-betrouwbaarheidsopmerking.** Dit notebook documenteert twee huidige beperkingen van Jenners PROC FORECAST (gap-test `400979`): de voorspelhorizon-`ID` wordt per stap met één eenheid verhoogd in plaats van met `INTERVAL=MONTH`, dus de afgedrukte voorspeldatums zijn niet de bedoelde kalendermaanden van 2025 (we bekijken de horizon daarom per stap); en `OUTRESID`/`OUTALL` vullen de één-stap-vooruit `_TYPE_='RESIDUAL'`-rijen nog niet, dus residudiagnostiek wordt vervangen door een directe vergelijking van de twee modellen. De puntvoorspellingen en betrouwbaarheidsgrenzen zelf worden wel geproduceerd en zijn waarop het bovenstaande verhaal is gebaseerd.